# Estado de Bell Φ⁺ — simulador y QPU real de IBM

Taller para la charla **Ciberseguridad en la era cuántica**. Ejecuta las celdas en orden. El circuito prepara $|\Phi^+\rangle=(|00\rangle+|11\rangle)/\sqrt{2}$.

> Seguridad: no pegues una API key en una celda. Usa Colab Secrets y revoca/rota una clave temporal al terminar.

## 1. Instalar dependencias
Si Colab lo solicita después, reinicia la sesión y continúa desde la siguiente celda.

In [ ]:
!pip install -q 'qiskit[visualization]' qiskit-aer qiskit-ibm-runtime

## 2. Importar y verificar versiones

In [ ]:
import qiskit, qiskit_aer, qiskit_ibm_runtime
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

print('qiskit:', qiskit.__version__)
print('qiskit-aer:', qiskit_aer.__version__)
print('qiskit-ibm-runtime:', qiskit_ibm_runtime.__version__)

## 3. Construir el estado de Bell Φ⁺

1. Iniciamos en $|00\rangle$.
2. H sobre q₀ crea superposición.
3. CX correlaciona q₁ con q₀.
4. Medimos ambos qubits.

Esperamos aproximadamente 50% `00` y 50% `11`.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)       # Superposición
bell.cx(0, 1)   # Entrelazamiento
bell.measure_all()
bell.draw('mpl')

## 4. Ejecutar primero en el simulador local
Este checkpoint no requiere cuenta, API key ni cola.

In [ ]:
simulator = AerSimulator()
compiled = transpile(bell, simulator)
result_sim = simulator.run(compiled, shots=1024).result()
counts_sim = result_sim.get_counts()

print(counts_sim)
plot_histogram(counts_sim)

## 5. Preparar acceso seguro a IBM Quantum

1. Entra a https://quantum.cloud.ibm.com/ y crea/selecciona una instancia.
2. Desde Home/Dashboard crea una API key y cópiala.
3. En la barra lateral de Colab abre **Secrets** (🔑).
4. Crea `IBM_QUANTUM_API_KEY` y habilita acceso del notebook.
5. Opcional: guarda el CRN como `IBM_QUANTUM_CRN`.

> IBM considera una nube externa un entorno no confiable. No uses `save_account()` aquí ni publiques el secreto.

In [ ]:
from google.colab import userdata

api_key = userdata.get('IBM_QUANTUM_API_KEY')
service = QiskitRuntimeService(token=api_key)

# Si necesitas una instancia específica, reemplaza la línea anterior por:
# crn = userdata.get('IBM_QUANTUM_CRN')
# service = QiskitRuntimeService(token=api_key, instance=crn)

## 6. Elegir una QPU, transpilar y enviar
Guarda el Job ID. La cola y la cuota dependen de tu plan.

In [ ]:
backend = service.least_busy(operational=True, simulator=False)
print('Backend:', backend.name)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_bell = pm.run(bell)

sampler = Sampler(mode=backend)
job = sampler.run([isa_bell], shots=1024)
job_id = job.job_id()
print('Job ID:', job_id)

## 7. Recuperar y comparar los resultados
En una QPU real pueden aparecer `01` y `10` por ruido.

In [ ]:
pub_result = job.result()[0]
counts_qpu = pub_result.data.meas.get_counts()

print(counts_qpu)
plot_histogram(
    [counts_sim, counts_qpu],
    legend=['Simulador', 'QPU real'],
)

## 8. Recuperar un trabajo anterior (opcional)

In [ ]:
# Sustituye por un ID guardado si cerraste la sesión:
# old_job = service.job('TU_JOB_ID')
# old_counts = old_job.result()[0].data.meas.get_counts()
# plot_histogram(old_counts)

## Interpretación

- El simulador ideal produce solo `00` y `11` (con fluctuación estadística).
- La QPU puede producir algunos `01`/`10` por errores de puertas, lectura y decoherencia.
- Un histograma en la base Z muestra correlación compatible con Φ⁺, pero no certifica por sí solo el entrelazamiento.
- Si ves casi 50% `01` y 50% `10`, revisa si agregaste `x(1)`: eso prepara Ψ⁺.